<a href="https://colab.research.google.com/github/mohamedmorsy-ai/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedmorsy-ai/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:

# ==================================================
# ML-04: SEARCH INTELLIGENCE DATA CONTRACT
# CELL 1: UNIT OF ANALYSIS + TIME WINDOW
# ==================================================

# Install required libraries
%pip -q install -U duckdb huggingface_hub pandas scikit-learn

import duckdb
import pandas as pd
from getpass import getpass
from huggingface_hub import login
from IPython.display import display

# ==================================================
# 1. PROJECT INFORMATION
# ==================================================

print("=" * 55)
print("FLYRANK ML-04: DATA CONTRACT")
print("=" * 55)

print("\nSelected Lane:")
print("Refresh / Content Opportunity Scoring")

print("\nUnit of Analysis:")
print("One client-content pair per feature row.")

print("\nSource Table:")
print("fact_content_daily_performance")

print("\nFeature Window:")
print("March 1-31, 2026")

print("\nDecision Cutoff:")
print("March 31, 2026")

print("\nOutcome Window:")
print("April 1-30, 2026")

print("\nPrediction Target:")
print("At least 20% decline in average daily impressions.")

print("\nDeliberate Exclusion:")
print("Future data and label-derived information.")

print("\nBusiness Objective:")
print("Rank content pages for SEO review.")

# ==================================================
# 2. HUGGING FACE AUTHENTICATION
# ==================================================

print("\n" + "=" * 55)
print("HUGGING FACE AUTHENTICATION")
print("=" * 55)

print("\nEnter your Hugging Face READ Token below.")
print("The token will be hidden while typing.")

hf_token = getpass("\nEnter HF Token: ").strip()

if not hf_token:
    raise ValueError("Hugging Face token is required.")

try:
    login(
        token=hf_token,
        add_to_git_credential=False
    )
    print("\nAuthentication successful!")

finally:
    del hf_token

# ==================================================
# 3. INITIALIZE DUCKDB
# ==================================================

print("\nInitializing DuckDB...")

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    PROVIDER credential_chain
)
""")

print("DuckDB initialized successfully.")

# ==================================================
# 4. WAREHOUSE DATA SOURCES
# ==================================================

BASE = "hf://datasets/FlyRank/internship-warehouse"

FACT = (
    f"{BASE}/fact_content_daily_performance"
)

MARCH = (
    f"read_parquet('{FACT}/month=2026-03/*.parquet')"
)

APRIL = (
    f"read_parquet('{FACT}/month=2026-04/*.parquet')"
)

print("\nWarehouse paths configured.")
print("March partition: 2026-03")
print("April partition: 2026-04")

# ==================================================
# 5. LOAD MARCH 2026 DATA
# ==================================================

print("\nLoading March 2026 warehouse data...")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE march AS
SELECT *
FROM {MARCH}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""")

print("March data loaded successfully!")

# ==================================================
# 6. VERIFY DATA CONNECTION
# ==================================================

result = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM march
""").df()

print("\nDATA VERIFICATION")
print("=" * 55)

display(result)

assert result.loc[0, "total_rows"] > 0, (
    "No March records were loaded."
)

# ==================================================
# 7. FINAL STATUS
# ==================================================

print("\n" + "=" * 55)
print("ML-04 CELL 1 COMPLETED SUCCESSFULLY")
print("=" * 55)

print("Authentication: SUCCESS")
print("DuckDB Connection: READY")
print("March Data: LOADED")
print("April Outcome Source: CONFIGURED")
print("Data Contract: INITIALIZED")

print("\nReady to execute Cell 2.")


FLYRANK ML-04: DATA CONTRACT

Selected Lane:
Refresh / Content Opportunity Scoring

Unit of Analysis:
One client-content pair per feature row.

Source Table:
fact_content_daily_performance

Feature Window:
March 1-31, 2026

Decision Cutoff:
March 31, 2026

Outcome Window:
April 1-30, 2026

Prediction Target:
At least 20% decline in average daily impressions.

Deliberate Exclusion:
Future data and label-derived information.

Business Objective:
Rank content pages for SEO review.

HUGGING FACE AUTHENTICATION

Enter your Hugging Face READ Token below.
The token will be hidden while typing.

Enter HF Token: ··········

Authentication successful!

Initializing DuckDB...
DuckDB initialized successfully.

Warehouse paths configured.
March partition: 2026-03
April partition: 2026-04

Loading March 2026 warehouse data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March data loaded successfully!

DATA VERIFICATION


,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31



ML-04 CELL 1 COMPLETED SUCCESSFULLY
Authentication: SUCCESS
DuckDB Connection: READY
March Data: LOADED
April Outcome Source: CONFIGURED
Data Contract: INITIALIZED

Ready to execute Cell 2.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [3]:

display(Markdown("""
### 2. Data Contract: Field Classification

**Model Features — Exactly Five**

1. `impressions_march`: Total March search impressions.
2. `clicks_march`: Total March search clicks.
3. `ctr_march`: March clicks divided by impressions.
4. `avg_position_march`: March average search position, weighted by impressions where possible.
5. `observed_days_march`: Number of March days with available GSC measurements.

**Why These Features Are Safe**

All five features describe measurements available at the March 31 decision cutoff.

**Label / Proxy**

`decline_april_20pct`: A binary label derived from April versus March average daily search impressions.

This is an observed future-decline proxy, not proof that a refresh will be beneficial.

**Context Fields**

- `client_hash_id`: Client grouping and validation.
- `content_hash_id`: Content item identification.
- `report_date`: Time-window construction.
- `gsc_data_available`: Tracking availability.

**Excluded Fields**

- April performance as an input: future information.
- The outcome label itself: direct target leakage.
- Client and content IDs: identifiers, not predictive signals.
- Product-generated scores: circular decision signals.
- June sample: reserved final-month observations.
- Raw identifying information: excluded for privacy.

**Missingness Policy**

Unavailable tracking is not interpreted as zero traffic. Records are filtered using `gsc_data_available IS TRUE`.

**Evaluation Plan**

Use a client-grouped holdout and compare the honest model with a deliberately leaked model.
"""))

FEATURES = [
    "impressions_march",
    "clicks_march",
    "ctr_march",
    "avg_position_march",
    "observed_days_march"
]

print("Feature contract:")
for i, feature in enumerate(FEATURES, 1):
    print(f"{i}. {feature}")

assert len(FEATURES) == 5
print("\nContract validation passed.")



### 2. Data Contract: Field Classification

**Model Features — Exactly Five**

1. `impressions_march`: Total March search impressions.
2. `clicks_march`: Total March search clicks.
3. `ctr_march`: March clicks divided by impressions.
4. `avg_position_march`: March average search position, weighted by impressions where possible.
5. `observed_days_march`: Number of March days with available GSC measurements.

**Why These Features Are Safe**

All five features describe measurements available at the March 31 decision cutoff.

**Label / Proxy**

`decline_april_20pct`: A binary label derived from April versus March average daily search impressions.

This is an observed future-decline proxy, not proof that a refresh will be beneficial.

**Context Fields**

- `client_hash_id`: Client grouping and validation.
- `content_hash_id`: Content item identification.
- `report_date`: Time-window construction.
- `gsc_data_available`: Tracking availability.

**Excluded Fields**

- April performance as an input: future information.
- The outcome label itself: direct target leakage.
- Client and content IDs: identifiers, not predictive signals.
- Product-generated scores: circular decision signals.
- June sample: reserved final-month observations.
- Raw identifying information: excluded for privacy.

**Missingness Policy**

Unavailable tracking is not interpreted as zero traffic. Records are filtered using `gsc_data_available IS TRUE`.

**Evaluation Plan**

Use a client-grouped holdout and compare the honest model with a deliberately leaked model.


Feature contract:
1. impressions_march
2. clicks_march
3. ctr_march
4. avg_position_march
5. observed_days_march

Contract validation passed.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:

from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score

display(Markdown("""
### 3. Warehouse Verification and Leakage Experiment

This experiment performs exactly three contract-verification queries:

1. Verify the daily fact table grain.
2. Verify row counts and reporting dates.
3. Verify tracking availability using `IS TRUE`.

Next, it creates five March features, defines the April decline label, trains an honest model, and demonstrates direct target leakage.
"""))

# Verification Query 1: Grain
q1 = """
SELECT report_date, client_hash_id,
       content_hash_id, COUNT(*) AS n
FROM march
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
"""

grain = con.sql(q1).df()
print("\nQUERY 1 - Grain check")
print("Duplicate examples:", len(grain))
display(grain)

# Verification Query 2: Counts and date window
q2 = """
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT client_hash_id) AS clients,
       COUNT(DISTINCT content_hash_id) AS content_items,
       MIN(report_date) AS first_date,
       MAX(report_date) AS last_date
FROM march
"""

counts = con.sql(q2).df()
print("\nQUERY 2 - Counts and date span")
display(counts)

# Verification Query 3: Availability
q3 = """
SELECT COUNT(*) AS total_rows,
       COUNT(*) FILTER (
           WHERE gsc_data_available IS TRUE
       ) AS available_rows,
       COUNT(*) FILTER (
           WHERE gsc_data_available IS NOT TRUE
       ) AS unavailable_rows
FROM march
"""

availability = con.sql(q3).df()
print("\nQUERY 3 - Availability")
display(availability)

assert grain.empty, "Duplicate daily grain found."

# Check available position columns
schema = set(con.sql(
    "DESCRIBE SELECT * FROM march"
).df()["column_name"])

if "gsc_sum_position" in schema:
    position_expression = """
    SUM(gsc_sum_position) /
    NULLIF(SUM(gsc_impressions), 0)
    """
elif "gsc_avg_position" in schema:
    position_expression = """
    SUM(gsc_avg_position * gsc_impressions) /
    NULLIF(SUM(gsc_impressions), 0)
    """
else:
    raise ValueError("No supported GSC position column found.")

# Build March feature frame
march_features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_march,
    SUM(gsc_clicks) AS clicks_march,
    SUM(gsc_clicks) * 1.0 /
        NULLIF(SUM(gsc_impressions), 0) AS ctr_march,
    {position_expression} AS avg_position_march,
    COUNT(DISTINCT report_date) AS observed_days_march
FROM march
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
HAVING SUM(gsc_impressions) >= 100
   AND COUNT(DISTINCT report_date) >= 20
""").df()

# Aggregate April outcome, never as model features
april_outcomes = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_april,
    COUNT(DISTINCT report_date) AS observed_days_april
FROM {APRIL}
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
HAVING COUNT(DISTINCT report_date) >= 20
""").df()

# Match the same client-content pairs
data = march_features.merge(
    april_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Compare average daily impressions
march_daily = (
    data["impressions_march"] /
    data["observed_days_march"]
)

april_daily = (
    data["impressions_april"] /
    data["observed_days_april"]
)

data["decline_april_20pct"] = (
    april_daily <= 0.8 * march_daily
).astype(int)

X = data[FEATURES].copy()
X = X.fillna(X.median())
y = data["decline_april_20pct"]
groups = data["client_hash_id"]

assert len(X) > 0, "No eligible content rows."
assert y.nunique() == 2, "Both label classes are required."

print("\nFIVE-FEATURE FRAME")
print("Eligible content pairs:", len(data))
print("Positive-label rate:", f"{y.mean():.2%}")

display(
    data[FEATURES + ["decline_april_20pct"]].head(10)
)

# Hold out entire clients
splitter = GroupShuffleSplit(
    n_splits=10,
    test_size=0.25,
    random_state=42
)

valid_split = None
for train_idx, test_idx in splitter.split(X, y, groups):
    if (y.iloc[train_idx].nunique() == 2 and
        y.iloc[test_idx].nunique() == 2):
        valid_split = (train_idx, test_idx)
        break

assert valid_split is not None, "No valid grouped split."

train_idx, test_idx = valid_split

# Honest model
model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

model.fit(X.iloc[train_idx], y.iloc[train_idx])
honest_pred = model.predict(X.iloc[test_idx])

honest_acc = accuracy_score(
    y.iloc[test_idx], honest_pred
)
honest_balanced = balanced_accuracy_score(
    y.iloc[test_idx], honest_pred
)

# Deliberate target leakage
X_leaky = X.copy()
X_leaky["leaked_label_copy"] = y.to_numpy()

leaked_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)
leaked_model.fit(
    X_leaky.iloc[train_idx],
    y.iloc[train_idx]
)
leaked_pred = leaked_model.predict(
    X_leaky.iloc[test_idx]
)
leaked_acc = accuracy_score(
    y.iloc[test_idx], leaked_pred
)

print("\nLEAKAGE EXPERIMENT")
print(f"Honest accuracy: {honest_acc:.4f}")
print(f"Honest balanced accuracy: {honest_balanced:.4f}")
print(f"Leaked accuracy: {leaked_acc:.4f}")

# Remove leaked information
X_leaky.drop(columns=["leaked_label_copy"], inplace=True)
assert list(X_leaky.columns) == FEATURES

del X_leaky, leaked_model

print("\nLeakage removed.")
print("Retained honest accuracy:", round(honest_acc, 4))
print("ML-04 experiment completed.")



### 3. Warehouse Verification and Leakage Experiment

This experiment performs exactly three contract-verification queries:

1. Verify the daily fact table grain.
2. Verify row counts and reporting dates.
3. Verify tracking availability using `IS TRUE`.

Next, it creates five March features, defines the April decline label, trains an honest model, and demonstrates direct target leakage.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


QUERY 1 - Grain check
Duplicate examples: 0


,report_date,client_hash_id,content_hash_id,n



QUERY 2 - Counts and date span


,total_rows,clients,content_items,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


... (4 outputs left out)


FIVE-FEATURE FRAME
Eligible content pairs: 88941
Positive-label rate: 49.97%


,impressions_march,clicks_march,ctr_march,avg_position_march,observed_days_march,decline_april_20pct
0,10849.0,22.0,0.002028,8.049866,31,0
1,705.0,1.0,0.001418,5.863830,31,1
2,2099.0,1.0,0.000476,2.632206,31,1
3,288.0,0.0,0.000000,10.489583,30,1
4,3535.0,25.0,0.007072,3.151344,31,0
5,1558.0,3.0,0.001926,5.247112,31,1
6,2021.0,0.0,0.000000,7.853538,31,1
7,1257.0,0.0,0.000000,6.887828,31,1
8,724.0,1.0,0.001381,22.888122,31,1
9,1276.0,1.0,0.000784,5.619122,31,1



LEAKAGE EXPERIMENT
Honest accuracy: 0.3892
Honest balanced accuracy: 0.5169
Leaked accuracy: 1.0000

Leakage removed.
Retained honest accuracy: 0.3892
ML-04 experiment completed.


/tmp/ipykernel_2300/563566026.py:210: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  X_leaky.drop(columns=["leaked_label_copy"], inplace=True)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [5]:

display(Markdown("""
### 4. Data Limitations

**1. Unbalanced Historical Coverage**

Different clients have different tracking start dates and available historical periods. Therefore, their observations may not be directly comparable.

**2. Missing Tracking Is Not Zero Traffic**

Unavailable Google Search Console measurements cannot be interpreted as zero search impressions or clicks.

**3. Seasonal and External Effects**

Performance changes may reflect seasonality, competition, search demand, or external algorithm changes.

**4. Outcome Proxy Limitations**

A decline of at least 20% in average daily impressions is a provisional research definition. It does not prove that a page requires a content refresh.

**5. Data Leakage**

April outcomes are unavailable at the March decision cutoff and must never be included among the five model features.

**6. Model Limitations**

The model supports content review prioritization. It cannot guarantee future rankings, prove causation, or replace human SEO judgment.

**Final Conclusion**

The data contract establishes a documented, leakage-aware foundation for the Refresh / Content Opportunity Scoring project.
"""))

print("ML-04 FINAL VALIDATION")
print("=" * 45)

print("Feature window: March 2026")
print("Outcome window: April 2026")
print("Model features:", len(FEATURES))
print("Duplicate grain examples:", len(grain))
print("Eligible content pairs:", len(data))
print("Honest model accuracy:", round(honest_acc, 4))
print("Leaked accuracy:", round(leaked_acc, 4))

assert len(FEATURES) == 5
assert grain.empty
assert "leaked_label_copy" not in FEATURES
assert X.columns.tolist() == FEATURES

print("\nALL FINAL CHECKS PASSED")
print("Ready for notebook review before submission.")



### 4. Data Limitations

**1. Unbalanced Historical Coverage**

Different clients have different tracking start dates and available historical periods. Therefore, their observations may not be directly comparable.

**2. Missing Tracking Is Not Zero Traffic**

Unavailable Google Search Console measurements cannot be interpreted as zero search impressions or clicks.

**3. Seasonal and External Effects**

Performance changes may reflect seasonality, competition, search demand, or external algorithm changes.

**4. Outcome Proxy Limitations**

A decline of at least 20% in average daily impressions is a provisional research definition. It does not prove that a page requires a content refresh.

**5. Data Leakage**

April outcomes are unavailable at the March decision cutoff and must never be included among the five model features.

**6. Model Limitations**

The model supports content review prioritization. It cannot guarantee future rankings, prove causation, or replace human SEO judgment.

**Final Conclusion**

The data contract establishes a documented, leakage-aware foundation for the Refresh / Content Opportunity Scoring project.


ML-04 FINAL VALIDATION
Feature window: March 2026
Outcome window: April 2026
Model features: 5
Duplicate grain examples: 0
Eligible content pairs: 88941
Honest model accuracy: 0.3892
Leaked accuracy: 1.0

ALL FINAL CHECKS PASSED
Ready for notebook review before submission.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.